In [ ]:
# Upgrade qdatoolkit to the latest version
# %pip install --upgrade qda-toolkit

# EXERCISE 2

The data in 'aircraft.csv' contains information about an industrial process for manufacturing aircraft components. The dataset includes data on the tensile Strength (kN) tested on produced parts, the Temperature (°C), Pressure (bar) and Time (minutes) of production.  

1. Fit a multiple linear regression model to predict product strength. 
2. Previous knowledge supports the hypothesis that the interaction between Temperature and Pressure could be important. Add the interaction to the model and verify its significance using a multiple linear regression model. 
3. In the absence of prior knowledge regarding which interaction terms may be significant, use forward selection stepwise regression to fit a multiple linear regression model. 
4. Compute the 95% confidence interval for the interaction coefficient between Temperature and Pressure in the regression model.

## 1. Fit a multiple linear regression model to predict product strength. 

In [ ]:
#Import the necessary libraries
import qdatoolkit as qda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
import statsmodels.api as sm

#Import the dataset
data = pd.read_csv('../Data/aircraft.csv')

# Inspect the dataset
data.head()

> Let's start by visualizing the data.

In [ ]:
# Plot 'Strength' against all other variables
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.scatter(data['Strength'], data['Temperature'], alpha=0.5)
plt.xlabel('Strength')
plt.ylabel('Temperature')
plt.subplot(1, 3, 2)
plt.scatter(data['Strength'], data['Pressure'], alpha=0.5)
plt.xlabel('Strength')
plt.ylabel('Pressure')
plt.subplot(1, 3, 3)
plt.scatter(data['Strength'], data['Time'], alpha=0.5)
plt.xlabel('Strength')
plt.ylabel('Time')
plt.show()

> It looks like there is a linear relationship between Time and Strength. The relationship between strength and pressure and temperature is not clear. 

> Let's fit a regression model including all potential regressors.

In [ ]:
# Fit the model including the interaction term
X = data[['Temperature', 'Pressure', 'Time']]
y = data['Strength']
X = sm.add_constant(X)  # Add a constant term to the predictor

model = sm.OLS(y, X).fit()

qda.summary(model)

- The regression in significant. 
- Temperature is not significant at 5%. 
- Pressure and time are significant. 

Let's proceed by checking assumptions on residuals.

In [ ]:
# Get the residuals
residuals = model.resid

In [ ]:
# Check the normality of the residuals
_ = qda.Assumptions(residuals).normality()

In [ ]:
# Check the randomness of the residuals
_ = qda.Assumptions(residuals).independence()

In [ ]:
#NORMALITY OF RESIDUALS
fig, axs = plt.subplots(2, 2)
fig.suptitle('Residual Plots')

axs[0,0].set_title('Normal probability plot')
stats.probplot(residuals, dist="norm", plot=axs[0,0])

axs[0,1].set_title('Versus Fits')
axs[0,1].scatter(model.fittedvalues, residuals)

fig.subplots_adjust(hspace=0.5)

axs[1,0].set_title('Histogram')
axs[1,0].hist(residuals)

axs[1,1].set_title('Time series plot')
axs[1,1].plot(np.arange(1, len(residuals)+1), residuals, 'o-')

## 2. Previous knowledge supports the hypothesis that the interaction between Temperature and Pressure could be important. Add the interaction to the model and verify its significance using a multiple linear regression model. 

In [ ]:
data['Temperature*Pressure'] = data['Temperature'] * data['Pressure']

# Fit the model including the interaction term
X = data[['Temperature', 'Pressure', 'Time', 'Temperature*Pressure']]
y = data['Strength']
X = sm.add_constant(X)  # Add a constant term to the predictor

model = sm.OLS(y, X).fit()

qda.summary(model)

In [ ]:
# Get the residuals
residuals = model.resid

In [ ]:
# Check the normality of the residuals
_ = qda.Assumptions(residuals).normality()

In [ ]:
# Check the randomness of the residuals
_ = qda.Assumptions(residuals).independence()

The normality assumption cannot be rejected. The model is validated and the interaction is significant. 

In [ ]:
#NORMALITY OF RESIDUALS
fig, axs = plt.subplots(2, 2)
fig.suptitle('Residual Plots')

axs[0,0].set_title('Normal probability plot')
stats.probplot(residuals, dist="norm", plot=axs[0,0])

axs[0,1].set_title('Versus Fits')
axs[0,1].scatter(model.fittedvalues, residuals)

fig.subplots_adjust(hspace=0.5)

axs[1,0].set_title('Histogram')
axs[1,0].hist(residuals)

axs[1,1].set_title('Time series plot')
axs[1,1].plot(np.arange(1, len(residuals)+1), residuals, 'o-')

## 3. In the absence of prior knowledge regarding which interaction terms may be significant, use forward selection stepwise regression to fit a multiple linear regression model. 

In [ ]:
# Compute all potential interaction terms
data['Temperature*Pressure'] = data['Temperature'] * data['Pressure']
data['Temperature*Time'] = data['Temperature'] * data['Time']
data['Pressure*Time'] = data['Pressure'] * data['Time']

# Create X and y for the new model
X = data[['Temperature', 'Pressure', 'Time', 'Temperature*Pressure', 'Temperature*Time', 'Pressure*Time']]
X = sm.add_constant(X)  # Add a constant term to the predictor
y = data['Strength']

In [ ]:
# Create a StepwiseRegression object using the qda library
stepwise = qda.StepwiseRegression(add_constant = True, direction='forward')
# Fit the model
model = stepwise.fit(y, X)

In [ ]:
# Print the summary of the model
model = model.model_fit
qda.summary(model)

> The stepwise regression results are consistent with the previous model.

> 3. Compute the 95% confidence interval for the interaction term between Temperature and Pressure in the regression model.

In [ ]:
# Calculate the confidence interval
CI_beta3 = model.conf_int(alpha=0.05).loc['Temperature*Pressure']
# Print the confidence interval
print('The confidence interval for beta is [%.3f, %.3f]' % (CI_beta3[0], CI_beta3[1]))